# Generate synthetic dataset

In [1]:
import pathlib

import fakeitmakeit as fm
import numpy as np
import pandas as pd

data_dir = pathlib.Path("data/large_synthetic")
data_dir.mkdir(parents=True, exist_ok=True)

## Presentations

In [2]:
n = 300
n_supervisors = 100
n_moderators = 6

supervisors = pd.Series(list(set(fm.name() for _ in range(n_supervisors))))
moderators = supervisors.sample(n_moderators, replace=False).to_numpy()

participant_2 = supervisors.sample(n, replace=True).to_numpy()
participant_3 = supervisors.sample(n, replace=True).to_numpy()


def who_is_the_chair(row):
    if row.participant_2 in moderators:
        return row.participant_2
    if row.participant_3 in moderators:
        return row.participant_3
    return pd.Series(moderators).sample(1, replace=True).iloc[0]


for i, p in enumerate(participant_3):
    candidate = p
    while candidate == participant_2[i]:
        candidate = supervisors.sample(1, replace=True).to_numpy()[0]
    participant_3[i] = candidate

presentations = (
    fm.cohort(n)
    .assign(
        participant_1=pd.col("first_name") + " " + pd.col("last_name"),
        participant_2=participant_2,
        participant_3=participant_3,
        chair=lambda df: df.apply(who_is_the_chair, axis=1),
    )
    .drop_duplicates(subset=["participant_1"])
    .loc[:, ["participant_1", "participant_2", "participant_3", "chair"]]
    .rename_axis("id")
)

# Presentations itself no longer enforces any of this (the four roles are fully
# symmetric) - these are just generator-side choices, to keep the synthetic
# dataset looking like a plausible real conference rather than a stress test.
assert presentations.participant_1.is_unique
assert presentations.participant_2.eq(presentations.participant_3).sum() == 0
assert presentations.participant_1.eq(presentations.participant_2).sum() == 0
assert presentations.participant_1.eq(presentations.participant_3).sum() == 0

print(f"Number of presentations: {len(presentations)}")
print(f"Number of unique participant_2: {len(presentations.participant_2.unique())}")
print(f"Number of unique participant_3: {len(presentations.participant_3.unique())}")
print(f"Number of unique chairs: {len(presentations.chair.unique())}")

presentations.to_csv(data_dir / "presentations.csv", index=True)

presentations.sample(20)

Number of presentations: 296
Number of unique participant_2: 97
Number of unique participant_3: 93
Number of unique chairs: 6


,participant_1,participant_2,participant_3,chair
id,,,,
bvb93,Brandy Bradley,Ronald Morgan,Tyler Huang,Joshua Cannon
myh67,Marie Henry,Adriana Hansen,David Washington,John Hays
ylh647,Yong Hao,William Smith,Darius Martin,Patricia Sanchez
tm824,Timothy Mall,Anthony Harris,Brian Lopez,James Garcia
cc65,Chao Cai,Ronald Morgan,Mary Glass,Daniel Moran
mw58,Ming Wen,Chad Munoz,Melissa Munoz,John Hays
yl191,Yan Lei,Kelly Lopez,Amy Thompson,James Garcia
jzz34,Juan Zhang,Melinda Oconnor,Dana Morgan,Joshua Cannon
gl99,Guiying Lin,Laura Gonzales,Beth Garcia,Laura Williams


## Session start times

In [3]:
session_start_times = pd.DataFrame(
    {
        "session": [1, 2, 3, 4, 5, 6, 7, 8],
        "start_time": [
            "09:30",
            "10:15",
            "11:00",
            "11:45",
            "14:00",
            "14:45",
            "15:30",
            "16:15",
        ],
    }
).astype({"start_time": "datetime64[ns]"})
session_start_times = session_start_times.set_index("session").loc[:, "start_time"]
session_start_times = session_start_times.dt.time

session_start_times.to_csv(data_dir / "session-start-times.csv", index=True)

session_start_times

session
1    09:30:00
2    10:15:00
3    11:00:00
4    11:45:00
5    14:00:00
6    14:45:00
7    15:30:00
8    16:15:00
Name: start_time, dtype: object

## Unavailability table

In [4]:
# Availability exceptions table — one row per unavailability rule.
#
# Semantics of "day" and "session" (both nullable, using pandas' Int64):
#   - day = <value>, session = NaN      -> unavailable ALL DAY on that day
#   - day = NaN,      session = <value> -> unavailable during that session, EVERY day
#   - day = <value>,  session = <value> -> unavailable for that specific slot only
#   - day = NaN,      session = NaN     -> unavailable for the ENTIRE conference
#
# NaN acts as a wildcard meaning "applies to all values" for that column.
#
# Note: "global" (entire-conference) rules are deliberately not generated here. At
# this scale, a single person with a global restriction who is also required (as
# participant or chair) for any presentation makes that presentation permanently
# unschedulable, and with ~100 supervisors shared across ~292 presentations that's
# virtually guaranteed to happen - which would make this whole fixture infeasible.
# The dedicated small `unavailable_entire_conference` scenario under tests/data
# exercises that semantic deterministically instead.

n_restrictions = 60
rng = np.random.default_rng()

# Allow repeated people so one person can have multiple restrictions.
people = supervisors.sample(n_restrictions, replace=True).to_numpy()

# Mixed rule types (day-only, session-only, specific-slot), each equally plausible.
rule_types = rng.choice(
    ["day_only", "session_only", "specific_slot"],
    size=n_restrictions,
    p=[0.35, 0.35, 0.30],
)

days = np.full(n_restrictions, np.nan)
sessions = np.full(n_restrictions, np.nan)

for i, rule in enumerate(rule_types):
    if rule == "day_only":
        days[i] = rng.integers(1, 8)
    elif rule == "session_only":
        sessions[i] = rng.integers(1, 9)
    elif rule == "specific_slot":
        days[i] = rng.integers(1, 8)
        sessions[i] = rng.integers(1, 9)

unavailable = pd.DataFrame(
    {
        "person": people,
        "day": days,
        "session": sessions,
    }
).astype({"day": "Int64", "session": "Int64"})

print(f"Number of restrictions: {len(unavailable)}")
repeat_counts = unavailable.person.value_counts()
print(f"People with multiple restrictions: {(repeat_counts > 1).sum()}")

unavailable.to_csv(data_dir / "unavailable.csv", index=False)

unavailable.sample(5)

Number of restrictions: 60
People with multiple restrictions: 13


,person,day,session
59,Mary King,4,<NA>
52,Douglas Kennedy,7,2
4,Megan Mcintyre,<NA>,7
54,Whitney Palmer,4,2
31,Anthony Harris,<NA>,7
